In [158]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes 
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import time


In [160]:
X,y=load_diabetes(return_X_y=True)

In [162]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [164]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)

In [166]:
lr=LinearRegression()

In [168]:
lr.fit(X_train,y_train)

LinearRegression()

In [170]:
print(lr.coef_)
print(lr.intercept_)

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


In [172]:
y_pred = lr.predict(X_test)

In [174]:
R2_score=r2_score(y_test,y_pred)

In [176]:
print(R2_score)

0.4399338661568969


In [178]:
class GDRegressor:

    def __init__(self,learning_rate=0.01,epochs=100):
        
        self.coef_= None
        self.intercept_=None
        self.lr=learning_rate
        self.epochs=epochs
        
    def fit(self,x_train,y_train):
        #init your coefs
        self.intercept_= 0
        self.coef_=np.ones(x_train.shape[1])
        
        for i in range(self.epochs):
            #update all the coef and intercept
            y_hat = np.dot(x_train,self.coef_) +self.intercept_

             # Intercept gradient
            intercept_der = -2* np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.lr * intercept_der)

             # Coefficient gradient
            coef_der = -2 *np.dot((y_train - y_hat),x_train)/x_train.shape[0]  
            self.coef_ =self.coef_ -(self.lr * coef_der)

        print(self.intercept_,self.coef_)

     
        
    def predict(self,X_test):
        return np.dot(X_test,self.coef_)+ self.intercept_
        

In [180]:
class SGDRegressor:
    def __init__(self,learning_rate=0.01,epochs=100):
        
         self.intercept_=None
         self.coef_=None
         self.lr=learning_rate
         self.epochs=epochs
        
    def fit(self,X_train,y_train):
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):
            for j in range(X_train.shape[0]):
                idx = np.random.randint(0,X_train.shape[0])

                y_hat = np.dot(X_train[idx],self.coef_) + self.intercept_
                
                intercept_der = -2* (y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)
                
                coef_der = -2  * np.dot((y_train[idx] - y_hat),X_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)
                
        print(self.intercept_,self.coef_)

     
    def predict(self,X_test):
        return np.dot(X_test,self.coef_)+ self.intercept_
        

In [182]:
import random
class MBGDRegressor:

    def __init__(self,batch_size,learning_rate=0.01,epochs=100):
        
        self.coef_= None
        self.intercept_=None
        self.lr=learning_rate
        self.epochs=epochs
        self.batch_size=batch_size
        
    def fit(self,x_train,y_train):
        #init your coefs
        self.intercept_= 0
        self.coef_=np.ones(x_train.shape[1])
        
        for i in range(self.epochs):
            for j in range(int(X_train.shape[0]/self.batch_size)):
                #update all the coef and intercept
                idx = random.sample(range(X_train.shape[0]),self.batch_size)
                
                y_hat = np.dot(x_train[idx],self.coef_) +self.intercept_

                # Intercept gradient
                intercept_der = -2* np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                # Coefficient gradient
                coef_der = -2 *np.dot((y_train[idx] - y_hat),X_train[idx])  
                self.coef_ =self.coef_ -(self.lr * coef_der)

        print(self.intercept_,self.coef_)

     
        
    def predict(self,X_test):
        return np.dot(X_test,self.coef_)+ self.intercept_
        

In [254]:
MB =MBGDRegressor(batch_size=int(X_train.shape[0]/10),learning_rate=0.1,epochs=20)

In [256]:
gdr =  GDRegressor(learning_rate =0.6,epochs = 1000)
sdr =  SGDRegressor(learning_rate =0.01,epochs = 50)

In [258]:
start =time.time()
gdr.fit(X_train,y_train)
print("Time taken:",time.time() - start)


151.98879593353988 [   7.81553013 -186.49510734  506.45449714  330.72279448  -45.79443697
 -123.85023287 -193.69457562   98.53559341  470.95977477   88.65520161]
Time taken: 0.08054423332214355


In [260]:
start =time.time()
sdr.fit(X_train,y_train)
print("Time taken:",time.time() - start)

160.56739711408366 [  57.09902362  -59.63830595  355.33049695  249.17979448   18.40155095
  -30.00397125 -172.00299706  128.12980082  319.61441547  130.4369573 ]
Time taken: 0.7829880714416504


In [261]:
start =time.time()
MB.fit(X_train,y_train)
print("Time taken:",time.time() - start)

151.28582914513817 [  17.09741548 -200.87971598  529.46663634  349.05077134  -57.63268355
 -144.8992458  -193.81587012  103.23347554  498.96476988   88.9852803 ]
Time taken: 0.04349684715270996


In [264]:
y_pred = gdr.predict(X_test)

In [266]:

y_pred1=sdr.predict(X_test)

In [268]:
y_pred2 = MB.predict(X_test)

In [270]:
print(r2_score(y_test,y_pred))
print(r2_score(y_test,y_pred1))
print(r2_score(y_test,y_pred2))

0.4520736256827783
0.41986830730763447
0.4450658622303172


In [792]:
from sklearn.linear_model import SGDRegressor

In [794]:
reg = SGDRegressor(learning_rate='constant',eta0=0.1)

In [796]:
reg.fit(X_train,y_train)

SGDRegressor(eta0=0.1, learning_rate='constant')

In [798]:
y_pred3= reg.predict(X_test)

In [841]:
r2_score(y_test,y_pred3)

0.448294597351797

In [923]:
sgd = SGDRegressor(learning_rate='constant',eta0=0.1)

In [935]:
batch_size=20


for i in range(100):
    idx = random.sample(range(X_train.shape[0]),batch_size)
    sgd.partial_fit(X_train[idx],y_train[idx])
      


In [937]:
print(sgd.intercept_)

[152.27877515]


In [939]:
sgd.coef_


array([  25.42159784, -109.92331463,  419.33992118,  289.98482737,
        -10.47130467,  -68.73559866, -187.5674513 ,  114.78242977,
        371.16123419,  125.25165047])

In [941]:
y_pred4 = sgd.predict(X_test)

In [943]:
r2_score(y_test,y_pred4)

0.45044773118189196